#### Name: Nguyen Thi Thu Trang
#### Class: DSEB65A
#### Student ID: 11230595
#### Homework 4

In [22]:
import pandas as pd

import numpy as np

import re

In [23]:
# Simulated financial data for analysis exercises (30 observations)

data_stock = {

    'Date': ['2023-10-02', '2023-10-03', 'Oct 04, 2023', '2023/10/05', '2023-10-06',

             '2023-10-09', '2023-10-10', '2023-10-11', 'INVALID', '2023-10-13',

             '2023-10-16', '2023-10-17', '2023-10-18', 'Oct 19, 2023', '2023/10/20',

             '2023-10-23', '2023-10-24', '2023-10-25', '2023-10-26', 'NOT A DATE',

             '2023-10-27', '2023-10-30', '2023-10-31', 'Nov 01, 2023', '2023/11/02',

             '2023-11-03', '2023-11-06', '2023-11-07', '2023-11-08', '2023-11-09'],

    'Close': [150.5, 152.0, 151.8, 153.2, 155.0, 154.5, 157.3, 156.9, 158.0, 159.2,

              160.1, 158.8, 161.5, 162.3, 161.0, 163.4, 165.1, 164.2, 163.0, 166.0,

              166.5, 168.2, 167.9, 170.1, 169.5, 172.3, 171.0, 173.5, 175.0, 174.2],

    'Volume': ['1.2M', '1,500,000', '1.1M', '1.8M', '2,100,000', '1.4M', '1.9M', '1,600,000', '1.3M', '2.5M',

               '1.7M', '1,900,000', '2.2M', '2.0M', '1,800,000', '2.3M', '2.6M', '2,400,000', '2.1M', '2.0M',

               '2.8M', '3.1M', '2,900,000', '3.5M', '3.2M', '3,800,000', '3.3M', '4.1M', '4,500,000', '4.2M'],

    'News_Headline': [

        'Tech stock rises on new product announcement',

        'Market remains steady, analysts are neutral',

        'Minor gain as company beats expectations by 2%',

        'Stock falls unexpectedly after competitor news',

        'Massive 5% surge following successful product launch',

        'Price drop of 1.5% due to market correction',

        'Stock is up again, continuing bullish trend',

        'Company announces new CEO; stock price stable',

        'Analysts predict a downturn for the tech sector',

        'Strong quarterly earnings report leads to a price gain',

        'New partnership announced, stock rises 1%',

        'Profit taking leads to a slight drop in price',

        'Successful new feature release causes price surge',

        'Market correction continues, stock down 1.2%',

        'Neutral outlook from major investment bank',

        'Stock gains momentum ahead of earnings call',

        'Price hits 3-month high, a 4% gain this week',

        'Slight pullback after a week of gains',

        'Competitor product launch fails, our stock is up',

        'Another successful day for the tech giant',

        'Investors optimistic, stock price continues to climb',

        'Major acquisition rumors cause a 3.5% surge',

        'End of month rally pushes stock higher',

        'New month starts with a strong gain',

        'Stock falls slightly on profit-taking',

        'Another successful product launch boosts confidence',

        'Bullish sentiment continues into the new week',

        'Record sales figures announced, stock up 6%',

        'Company expands into new market, price rises',

        'Market sentiment turns neutral before holiday'

    ]

}

df_stock = pd.DataFrame(data_stock)

In [24]:
df_stock.head()

,Date,Close,Volume,News_Headline
0,2023-10-02,150.5,1.2M,Tech stock rises on new product announcement
1,2023-10-03,152.0,"1,500,000","Market remains steady, analysts are neutral"
2,"Oct 04, 2023",151.8,1.1M,Minor gain as company beats expectations by 2%
3,2023/10/05,153.2,1.8M,Stock falls unexpectedly after competitor news
4,2023-10-06,155.0,"2,100,000",Massive 5% surge following successful product ...


##### **Exercise 1: In-depth Data Cleaning and Standardization**

In [25]:
# a
df_stock['Date'] = pd.to_datetime(df_stock['Date'], format='mixed', errors='coerce')
df_stock = df_stock.dropna(subset=['Date'])

In [26]:
# b 
def convert_volumn(vol: str):
    vol = vol.replace(',', '')
    if 'M' in vol:
        return float(vol.replace('M', '')) * 1000000
    else:
        return float(vol)

df_stock['Volume'] = df_stock['Volume'].apply(convert_volumn)

##### **Exercise 2: Time Series Analysis - Setting Index and Calculating Price Change**

In [27]:
# a 
df_stock.set_index('Date', inplace=True)

In [28]:
# b 
df_stock['Price_Change'] = df_stock['Close'].diff()

##### **Exercise 3: Weekly Performance Analysis using Groupby**

In [29]:
# a 
df_stock['WeekofYear'] = df_stock.index.isocalendar().week

In [30]:
# b 
df_stock.groupby('WeekofYear')['Volume'].sum()

WeekofYear
40     7700000.0
41     7400000.0
42     9600000.0
43    12200000.0
44    16500000.0
45    16100000.0
Name: Volume, dtype: float64

In [31]:
df_stock.groupby('WeekofYear')['Close'].mean()

WeekofYear
40    152.500
41    156.975
42    160.740
43    164.440
44    169.600
45    173.425
Name: Close, dtype: float64

##### **Exercise 4: Monthly Analysis**

In [47]:
monthly_max_volume = df_stock.groupby(df_stock.index.month)['Volume'].max()

monthly_max_dates = df_stock.groupby(df_stock.index.month).apply(
    lambda x: x.index[x['Volume'] == x['Volume'].max()][0]
)

monthly_max_volume_dates = pd.DataFrame({
    'Max_Volume': monthly_max_volume,
    'Date_of_Max_Volume': monthly_max_dates
})
monthly_max_volume_dates

,Max_Volume,Date_of_Max_Volume
Date,,
10,3100000.0,2023-10-30
11,4500000.0,2023-11-08


##### **Exercise 5: News Sentiment Classification (Sentiment Analysis)**

In [57]:
positive = ['rises', 'beats expectations', 'surge', 'up', 'gain', 'successful']
negative = ['falls', 'drop', 'downturn', 'correction']

def sentiment_class(headline):
    if any(word.lower() in headline for word in positive):
        return 'Positive'
    elif any(word.lower() in headline for word in negative):
        return 'Negative'
    else:
        return 'Neutral'

df_stock['Sentiment'] = df_stock['News_Headline'].apply(sentiment_class)

##### **Exercise 6: Extracting Structured Event Information with Regex**

In [61]:
pattern = r'(?P<Action>surge?|rises?|gains?|falls?|drops?|down?)\D*(?P<Value>\d+\.?\d*)\s*(?P<Unit>\%)'
event_components = df_stock['News_Headline'].str.extract(pattern)
event_components = event_components.fillna('-')
event_components

,Action,Value,Unit
Date,,,
2023-10-02,-,-,-
2023-10-03,-,-,-
2023-10-04,gain,2,%
2023-10-05,-,-,-
2023-10-06,-,-,-
2023-10-09,drop,1.5,%
2023-10-10,-,-,-
2023-10-11,-,-,-
2023-10-13,-,-,-


##### **Exercise 7: Analyzing Correlation between Sentiment and Price Change**

In [62]:
df_stock.groupby('Sentiment')['Price_Change'].mean()

Sentiment
Negative   -0.040000
Neutral     0.128571
Positive    1.533333
Name: Price_Change, dtype: float64

##### **Exercise 8: Event Analysis - Finding the Day with the Largest Price Increase**

In [64]:
# a 
df_stock.loc[df_stock['Price_Change'] == df_stock['Price_Change'].max()]

,Close,Volume,News_Headline,Price_Change,WeekofYear,Sentiment
Date,,,,,,
2023-10-27,166.5,2800000.0,"Investors optimistic, stock price continues to...",3.5,43,Neutral


In [66]:
df_stock[df_stock['Price_Change'] == df_stock['Price_Change'].max()]['News_Headline'].values[0]

'Investors optimistic, stock price continues to climb'

##### **Exercise 9: Sentiment Analysis by Day of the Week**

In [ ]:
# a
df_stock['DayOfWeek'] = df_stock.index.day_name()

Date
2023-10-02       Monday
2023-10-03      Tuesday
2023-10-04    Wednesday
2023-10-05     Thursday
2023-10-06       Friday
2023-10-09       Monday
2023-10-10      Tuesday
2023-10-11    Wednesday
2023-10-13       Friday
2023-10-16       Monday
2023-10-17      Tuesday
2023-10-18    Wednesday
2023-10-19     Thursday
2023-10-20       Friday
2023-10-23       Monday
2023-10-24      Tuesday
2023-10-25    Wednesday
2023-10-26     Thursday
2023-10-27       Friday
2023-10-30       Monday
2023-10-31      Tuesday
2023-11-01    Wednesday
2023-11-02     Thursday
2023-11-03       Friday
2023-11-06       Monday
2023-11-07      Tuesday
2023-11-08    Wednesday
2023-11-09     Thursday
Name: DayOfWeek, dtype: object

In [ ]:
# b
sentiment_count = df_stock.groupby('DayOfWeek')['Sentiment'].value_counts().unstack(fill_value=0)
sentiment_count[sentiment_count['Negative'] == sentiment_count['Negative'].max()].index[0]

'Thursday'

##### **Exercise 10: Comprehensive Problem - Assessing the Impact of "Product Launch"**

In [90]:
# a
product_launch_dates = df_stock[
    df_stock['News_Headline'].str.contains('product launch', case=False)
].index

product_launch_dates

DatetimeIndex(['2023-10-06', '2023-10-26', '2023-11-03'], dtype='datetime64[ns]', name='Date', freq=None)

In [95]:
# b 
next_day_changes = df_stock.loc[product_launch_dates]['Price_Change'].shift(-1)
print(next_day_changes)

Date
2023-10-06   -1.2
2023-10-26    2.8
2023-11-03    NaN
Name: Price_Change, dtype: float64


In [96]:
# c 
avg_next_day_changes = df_stock.loc[product_launch_dates]['Price_Change'].shift(-1).mean()
print(avg_next_day_changes)

0.8000000000000114
